# Prithvi EO 2.0 — Feature Extraction

This notebook extracts patch embeddings from Prithvi and visualizes them using PCA and UMAP to understand the learned representation space.

Patch embeddings from the final transformer layer capture high-level semantic information about land cover, phenology, and surface properties. Visualizing these embeddings reveals how well the model separates different land cover types without any fine-tuning.

## References
- Prithvi HuggingFace: https://huggingface.co/ibm-nasa-geospatial

## 1. Setup

In [ ]:
import sys
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModel

sys.path.insert(0, os.path.abspath("../../.."))
try:
    from dotenv_config import HF_TOKEN
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

model_id = "ibm-nasa-geospatial/Prithvi-EO-2.0-300M"

model = AutoModel.from_pretrained(
    model_id, token=HF_TOKEN or None, trust_remote_code=True
)
model.eval()
print("Model loaded.")

## 2. Simulate Multi-Class EO Dataset

In a real workflow, load actual HLS chips with known land cover labels. Here we simulate chips from 5 land cover classes with distinct spectral signatures.

In [ ]:
# Simulate HLS reflectance for 5 land cover classes
# Bands: Blue, Green, Red, NIR, SWIR-1, SWIR-2 (scaled to 0-10000)
class_signatures = {
    "Water":       [200,  300,  250,  150,  100,   80],
    "Forest":      [300,  600,  400, 4000,  900,  500],
    "Cropland":    [400,  900,  700, 3500, 1800, 1000],
    "Urban":       [900, 1000, 1100, 1200, 1300, 1200],
    "Bare soil":   [700,  800,  900, 1100, 1500, 1400],
}

n_per_class = 20
img_size = 224
num_frames = 3

all_chips = []
all_labels = []
class_names = list(class_signatures.keys())

for label_idx, (cls_name, sig) in enumerate(class_signatures.items()):
    for _ in range(n_per_class):
        # Create a chip with the class spectral signature + noise
        sig_arr = np.array(sig, dtype=np.float32) / 10000.0  # normalize
        chip = np.random.normal(
            loc=sig_arr[None, :, None, None],
            scale=0.02,
            size=(num_frames, 6, img_size, img_size),
        ).astype(np.float32)
        all_chips.append(chip)
        all_labels.append(label_idx)

chips_tensor = torch.tensor(np.stack(all_chips))
labels_array = np.array(all_labels)
print(f"Dataset: {chips_tensor.shape} ({n_per_class} chips x {len(class_names)} classes)")

## 3. Extract Embeddings

In [ ]:
embeddings_list = []
batch_size = 4

with torch.no_grad():
    for i in range(0, len(chips_tensor), batch_size):
        batch = chips_tensor[i:i+batch_size]
        out = model(pixel_values=batch)
        # Mean-pool over patch tokens to get a single vector per image
        emb = out.last_hidden_state.mean(dim=1)  # [batch, embed_dim]
        embeddings_list.append(emb.numpy())

embeddings = np.vstack(embeddings_list)
print(f"Embeddings shape: {embeddings.shape}  ({len(embeddings)} samples x embedding_dim)")

## 4. PCA Visualization

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
emb_pca = pca.fit_transform(embeddings)

colors = plt.cm.tab10(np.linspace(0, 1, len(class_names)))

fig, ax = plt.subplots(figsize=(8, 7))
for idx, cls_name in enumerate(class_names):
    mask = labels_array == idx
    ax.scatter(emb_pca[mask, 0], emb_pca[mask, 1], c=[colors[idx]], label=cls_name, s=60, alpha=0.8)

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
ax.set_title("PCA of Prithvi EO 2.0-300M patch embeddings")
ax.legend()
plt.tight_layout()
plt.show()

## 5. UMAP Visualization (optional — install umap-learn)

In [ ]:
try:
    import umap

    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=10, min_dist=0.1)
    emb_umap = reducer.fit_transform(embeddings)

    fig, ax = plt.subplots(figsize=(8, 7))
    for idx, cls_name in enumerate(class_names):
        mask = labels_array == idx
        ax.scatter(emb_umap[mask, 0], emb_umap[mask, 1], c=[colors[idx]], label=cls_name, s=60, alpha=0.8)
    ax.set_title("UMAP of Prithvi EO 2.0-300M patch embeddings")
    ax.legend()
    plt.tight_layout()
    plt.show()

except ImportError:
    print("umap-learn not installed. To install: uv add umap-learn")
    print("UMAP often gives better cluster separation than PCA for high-dim embeddings.")

## 6. Linear Probe Accuracy

A linear classifier on frozen Prithvi embeddings shows how much the pretrained features separate land cover classes without any fine-tuning of the backbone.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

clf = LogisticRegression(max_iter=1000, random_state=42)
scores = cross_val_score(clf, embeddings, labels_array, cv=5, scoring="accuracy")

print(f"5-fold linear probe accuracy: {scores.mean()*100:.1f}% ± {scores.std()*100:.1f}%")
print("\nA high linear probe accuracy indicates Prithvi embeddings are already")
print("well-separated by land cover class before any fine-tuning.")